# 02 — Data Exploration

Inspect downloaded PSG recordings and hypnograms for structural consistency, signal quality, and annotation coverage.

**Notebook Objectives**

- Verify channel composition and sampling rate consistency across records.
- Identify signal-level irregularities that may affect preprocessing.
- Confirm PSG–hypnogram temporal alignment for epoching.
- Document constraints that inform epoching and feature extraction.

**Contents**

1. Setup and imports
2. Load a sample sleep session
3. Inspect EDF structure and channel metadata
4. Plot sample signals and hypnograms
5. Summarize findings

**Inputs**

- Local PSG + hypnogram EDF files (from `01 — Data Fetch`)

**Outputs**

- Notes and diagnostics used to guide preprocessing and resampling decisions

In [ ]:
# --- Configure the Python path to allow local imports from the src under project root ---
import sys
from pathlib import Path

# add src folder parent (the Project root) to sys.path
sys.path.insert(0, str(Path.cwd().parent))
print("Added to sys.path:", str(Path.cwd().parent))

In [ ]:
# --- Local imports ---
from src import constants as const
from src import folders as fldr
from src import plotters as plot
from src import psg_reader as pr
from src import hypnogram_reader as hr

In [ ]:
# --- Retrieve sample sleep session file pair (EDF and hypnogram)  ---
file_pairs = fldr.get_edf_file_pairs()
print(f"Retrieved {len(file_pairs)} EDF file pairs") 

# Select a representative record for structural inspection.
# Assumes deterministic ordering from `get_edf_file_pairs`.
subject_id, psg_file, hypno_file = file_pairs[0]
print(f"Selected subject ID: {subject_id}")
print(f"Sample EDF file: {psg_file}")
print(f"Sample hypnogram file: {hypno_file}")

In [ ]:
# --- Read sample EDF file and explore its contents ---
edf_data = pr.PsgReader(psg_file)

# Structural validation:
# Inspect channel count, sampling rates, scaling, and duration
# to confirm consistency with expected Sleep-EDF characteristics.
print("--- PSG File Summary ---")
print("Number of channels:", edf_data.no_of_channels)
print("Number of data blocks:", edf_data.no_of_data_blocks)
print("Duration per data block (seconds):", edf_data.duration_of_data_block)
print("Recording start time (s since epoch):", edf_data.recording_start_time_s)
print("Recording duration (seconds):", edf_data.recording_duration)
print("Channels names:", edf_data.channel_names)
print("Channel scale:", edf_data.channel_scale)
print("Channel low pass filter (Hz):", edf_data.channel_lpf)
print("Channel high pass filter (Hz):", edf_data.channel_hpf)
print("Channel sample count:", edf_data.ch_sample_count)
print("Channel sampling rate (Hz):", edf_data.ch_sampling_rate)


In [ ]:
# --- Sample plot of a signal channel ---
# Plot a short temporal window to verify amplitude scale,
# waveform continuity, and absence of obvious artifacts.
channel_data = edf_data.get_channel_data(channel=const.CHANNEL_SELECTION)
plot.plot_signal(
    x_data=channel_data.x_data,
    y_data=channel_data.y_data,
    x_label=channel_data.x_label,
    y_label=f"{const.CHANNEL_SELECTION} (uV)",
    y_scale=channel_data.y_scale_factor,
    title=f"Signal for channel {const.CHANNEL_SELECTION}",
    xlim_s=(0, 60),  # show only the first minute for clarity
)

In [ ]:
# --- Read sample hypnogram file and explore its contents ---
hypno_data = hr.HypnogramReader(hypno_file)

# Instance variables
print("--- Hypnogram Summary ---")
print("Recording duration:", hypno_data.recording_duration)
print("Unique stages:", hypno_data.unique_stages)

In [ ]:
# --- Sample plot of hypnogram data ---
map_config = hr.MappingConfig(
        enabled=False,
        remap={},  # no remapping; use original labels
    )
hypnogram_data = hypno_data.get_hypnogram_data(
    remap=map_config,
)
plot.plot_hypnogram_from_annotations(
    times=hypnogram_data.times,
    vals=hypnogram_data.vals,
    unique_labels=hypnogram_data.unique_labels,
    subject="Example Subject",
    xlim_s=(20000, 60000),
)